In [1]:
#Install matplotlib and restart kernel
%pip install matplotlib
%pip uninstall bokeh -y
%pip install bokeh==2.4.2
%pip install seaborn
%pip install "sagemaker>=2,<3"
%reset -f

# Install dependencies
import boto3
import io
import json
import math
import matplotlib.pyplot as plt
import os
import pandas as pd
import re
import sagemaker
import sys
import time
import zipfile

from sagemaker.debugger import Rule, rule_configs
from IPython.display import FileLink, FileLinks
from sagemaker import image_uris
from IPython.display import display
from IPython.display import Image
from sagemaker.analytics import ExperimentAnalytics
from sagemaker.inputs import TrainingInput
from sagemaker.session import Session
from sagemaker.tuner import IntegerParameter, CategoricalParameter, ContinuousParameter, HyperparameterTuner
from sagemaker.xgboost.estimator import XGBoost
from time import gmtime, strftime

sagemaker_session = sagemaker.Session()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
sess = boto3.Session()
sm = sess.client('sagemaker')

In [3]:
# Import the datasets
s3 = boto3.resource('s3')
for buckets in s3.buckets.all():
    if 'labdatabucket' in buckets.name:
        bucket = buckets.name
print("Bucket: ", bucket)
prefix = 'scripts/data'
output_path = 's3://{}/{}/output'.format(bucket, prefix)

# Configure the training paths
train_path = f"s3://{bucket}/{prefix}/adult_data_processed_train.csv"
validation_path = f"s3://{bucket}/{prefix}/adult_data_processed_validation.csv"

# Set up the TrainingInput objects
train_input = TrainingInput(train_path, content_type='text/csv')
validation_input = TrainingInput(validation_path, content_type='text/csv')

# Print the training and validation paths
print(f'Training path: {train_path}')
print(f'Validation path: {validation_path}')

# Set the container, name, and tags
create_date = strftime("%m%d%H%M")
container = image_uris.retrieve(framework='xgboost',region=boto3.Session().region_name,version='1.5-1')
run_name = 'lab-2-run-{}'.format(create_date)

In [4]:
xgb_model = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role, 
    instance_count=1, 
    instance_type='ml.m5.xlarge',
    output_path=output_path,
    sagemaker_session=sagemaker_session,
    rules=[
        Rule.sagemaker(
            rule_configs.create_xgboost_report(),
            rule_parameters={
                "save_interval": "5"
            }
        )
    ]
)

In [5]:
xgb_model.set_hyperparameters(
    max_depth=5,
    eta=0.1,  # Slower learning
    gamma=4,
    min_child_weight=6,
    subsample=0.7,
    verbosity=1,
    objective='binary:logistic',
    num_round=1000  # More rounds
)

In [6]:
xgb_model.fit(
    {
        "train": train_input,
        "validation": validation_input
    },
    wait=True
)

In [7]:
bucket, project_prefix = xgb_model.output_path[5:].split('/',1)

In [9]:
import os
import numpy as np
import xgboost as xgb
import tarfile
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print('Generating inline evaluation report...\n')

s3_client = boto3.client('s3')

# Download and extract the trained model
model_artifact_key = f"{project_prefix}/{xgb_model.latest_training_job.job_name}/output/model.tar.gz"
s3_client.download_file(bucket, model_artifact_key, 'model.tar.gz')
with tarfile.open('model.tar.gz', 'r:gz') as tar:
    tar.extractall(filter='data')

model = xgb.Booster()
model.load_model('xgboost-model')

# Load test data
test_path = f"s3://{bucket}/scripts/data/adult_data_processed_test.csv"
test_df = pd.read_csv(test_path, header=None)
y_test = test_df.iloc[:, 0].values
X_test = test_df.iloc[:, 1:].values

# Generate predictions
dtest = xgb.DMatrix(X_test)
y_pred_prob = model.predict(dtest)
y_pred = (y_pred_prob >= 0.5).astype(int)

# Confusion Matrix
print('=' * 50)
print('CONFUSION MATRIX')
print('=' * 50)
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['<=50K (0)', '>50K (1)'],
            yticklabels=['<=50K (0)', '>50K (1)'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

# Classification Report
print('\n' + '=' * 50)
print('CLASSIFICATION REPORT')
print('=' * 50 + '\n')
print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))
print(f'Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}')

# Feature Importance
print('\n' + '=' * 50)
print('FEATURE IMPORTANCE (Top 20 by Gain)')
print('=' * 50)
feature_names = [
    'age', 'education', 'education_num', 'occupation', 'capital_gain',
    'capital_loss', 'hours_per_week', 'marital_Married-civ-spouse',
    'marital_Never-married', 'marital_Divorced', 'marital_Separated',
    'marital_Widowed', 'marital_Married-spouse-absent', 'marital_Married-AF-spouse',
    'race_White', 'race_Black', 'race_Asian-Pac-Islander', 'race_Amer-Indian-Eskimo',
    'race_Other', 'relationship_Husband', 'relationship_Not-in-family',
    'relationship_Own-child', 'relationship_Unmarried', 'relationship_Wife',
    'relationship_Other-relative', 'sex_Male', 'sex_Female',
    'workclass_Private', 'workclass_Self-emp-not-inc', 'workclass_Local-gov',
    'workclass_?', 'workclass_State-gov', 'workclass_Self-emp-inc',
    'workclass_Federal-gov'
]
importance = model.get_score(importance_type='gain')
imp_df = pd.DataFrame({
    'Feature': [feature_names[int(k[1:])] if k.startswith('f') and int(k[1:]) < len(feature_names) else k
                for k in importance.keys()],
    'Importance': list(importance.values())
})
imp_df = imp_df.sort_values('Importance', ascending=True).tail(20)
plt.figure(figsize=(10, 8))
plt.barh(imp_df['Feature'], imp_df['Importance'])
plt.title('Feature Importance (Top 20 by Gain)')
plt.xlabel('Gain')
plt.tight_layout()
plt.show()